# FIFA Rankings vs ELO Ratings
## Predictive Validity in World Cup Knockout Stages (1994–2022)

---

> **The question this notebook answers:** When a stronger team faces a weaker team in a World Cup knockout match, which system — FIFA's official World Rankings or the ELO rating system — is better at identifying who the stronger team actually is?

> **Why it matters:** FIFA uses its own rankings to seed every World Cup draw. If those rankings are less accurate than a freely available alternative, tournament draws are systematically unfair.

> **What we find:** ELO outperforms FIFA on every single metric tested. The gap narrowed after FIFA adopted an ELO-style formula in 2018 — but it did not close.

---

### How to read this notebook

The analysis follows a deliberate narrative arc:

| Chapter | Question |
|---|---|
| **0 — Setup** | Load data, fit models once, verify numbers |
| **1 — The Data** | What does the dataset actually look like? |
| **2 — The Core Finding** | How much better is ELO? Six ways to measure it |
| **3 — Why McNemar's Test Failed** | The gap is real — the test just can't see it yet |
| **4 — Does the Gap Hold Over Time?** | Pre/post 2018 and tournament-by-tournament stability |
| **5 — Three Robustness Checks** | Staleness, 1994 exclusion, FIFA Points vs Rank |
| **6 — Summary** | Everything in one table |

**Run cells in order.** All models are fitted once in Chapter 0 and reused throughout.

---
## Chapter 0 — Setup & Model Fitting

Everything lives here. Run this first and run it once.

**Design choices explained here:**
- `y = home_won`: The logistic models predict whether the home-listed team wins. Since Elo Diff is signed (positive = home team stronger), this is mathematically equivalent to predicting the favourite wins.
- `fit_intercept=False`: All knockout matches are at neutral venues. Equal teams should give exactly 50% win probability. An intercept would corrupt this.
- `Pseudo-R² against 50/50 null`: Standard statsmodels pseudo-R² compares against an intercept-only baseline (home wins ~58%). The correct null at neutral venues is 50/50. We compute this manually.
- `Accuracy from Elo Correct column`: Not from model predictions. This cleanly measures "did the ELO-favoured team win?" with no model artefacts.

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.api as sm
import scipy.stats as scipy_stats
from scipy.stats import norm as norm_dist
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
warnings.filterwarnings('ignore')

# ── Working directory ──────────────────────────────────────────────────────
# os.chdir(r'D:\FIFA Research')   # uncomment if needed
print(f"Working directory: {os.getcwd()}")

# ── Visual style ───────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0', 'grid.linewidth': 0.8,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})
ELO_COLOR  = '#2563EB'   # blue
FIFA_COLOR = '#D97706'   # amber
NAVY       = '#1B3A6B'   # dark navy

# ── Load data ──────────────────────────────────────────────────────────────
EXCEL = 'WC_Knockout_Complete_Dataset.xlsx'
df_all = pd.read_excel(EXCEL, sheet_name='🏆 KO Matches (Main)', header=1)
df_rank = pd.read_excel(EXCEL, sheet_name='📈 FIFA Rankings Used',  header=1)
df_sq   = pd.read_excel(EXCEL, sheet_name='📋 Squad Lists',         header=1)

# ── Analysis sample: 115 matches (13 excluded — missing ELO or FIFA rank) ─
if 'In Analysis (115)' not in df_all.columns:
    df_all['In Analysis (115)'] = np.where(
        df_all[['Home Elo','Away Elo','Home Fifa Rank','Away Fifa Rank']].isna().any(axis=1),0,1)

df = df_all[df_all['In Analysis (115)'] == 1].copy()
df['Match Date']     = pd.to_datetime(df['Match Date'])
df['Wc Year Int']    = df['Wc Year Int'].astype(int)
df['Is Shootout']    = pd.to_numeric(df['Is Shootout'],   errors='coerce').fillna(0).astype(int)
df['Extra Time']     = pd.to_numeric(df['Extra Time'],    errors='coerce').fillna(0)
df['Days Stale']     = pd.to_numeric(df['Days Stale'],    errors='coerce')
df['Elo Diff']       = pd.to_numeric(df['Elo Diff'],      errors='coerce')
df['Fifa Rank Diff'] = pd.to_numeric(df['Fifa Rank Diff'],errors='coerce')
df['Fifa Pts Diff']  = pd.to_numeric(df['Fifa Pts Diff'], errors='coerce')
df['Elo Correct']    = pd.to_numeric(df['Elo Correct'],   errors='coerce').astype(int)
df['Fifa Correct']   = pd.to_numeric(df['Fifa Correct'],  errors='coerce').astype(int)
df['Goal Diff']      = pd.to_numeric(df['Goal Diff'],     errors='coerce')
df['home_won']       = (df['Actual Winner'] == df['Home Team']).astype(int)
df['elo_upset']      = (df['Elo Correct']  == 0).astype(int)
df['fifa_upset']     = (df['Fifa Correct'] == 0).astype(int)

ROUND_ORDER = {'Round of 16':1,'Quarter-finals':2,'Semi-finals':3,
               'Final':4,'Third Place':3,'Third-place match':3}
df['round_num']     = df['Round'].map(ROUND_ORDER)
df['year_centered'] = df['Wc Year Int'].astype(float) - df['Wc Year Int'].astype(float).mean()
df['post_2018']     = (df['Wc Year Int'] >= 2018).astype(int)

# ── Accuracy (computed from pre-built flags, not from model predictions) ───
elo_acc  = df['Elo Correct'].mean()   # 73.9%
fifa_acc = df['Fifa Correct'].mean()  # 68.7%

# ── Fit logistic models ONCE — no intercept, y = home_won ──────────────────
y      = df['home_won'].values
X_elo  = df[['Elo Diff']].values
X_fifa = df[['Fifa Rank Diff']].values
X_pts  = df[['Fifa Pts Diff']].values

model_elo  = LogisticRegression(fit_intercept=False, random_state=42).fit(X_elo,  y)
model_fifa = LogisticRegression(fit_intercept=False, random_state=42).fit(X_fifa, y)
model_pts  = LogisticRegression(fit_intercept=False, random_state=42).fit(X_pts,  y)

y_prob_elo  = model_elo.predict_proba(X_elo)[:,1]
y_prob_fifa = model_fifa.predict_proba(X_fifa)[:,1]
y_prob_pts  = model_pts.predict_proba(X_pts)[:,1]

sm_elo  = sm.Logit(y, X_elo).fit(disp=0)
sm_fifa = sm.Logit(y, X_fifa).fit(disp=0)

# ── Pseudo-R² against 50/50 null (correct for neutral venues) ─────────────
ll_null_50     = np.sum(y * np.log(0.5) + (1-y) * np.log(0.5))
pseudo_r2_elo  = 1 - sm_elo.llf  / ll_null_50
pseudo_r2_fifa = 1 - sm_fifa.llf / ll_null_50

# ── AUC and Brier ──────────────────────────────────────────────────────────
auc_elo   = roc_auc_score(y, y_prob_elo)
auc_fifa  = roc_auc_score(y, y_prob_fifa)
auc_pts   = roc_auc_score(y, y_prob_pts)
brier_elo  = np.mean((y_prob_elo  - y)**2)
brier_fifa = np.mean((y_prob_fifa - y)**2)

# ── OLS (goal difference, non-shootout matches only) ──────────────────────
df_reg   = df[df['Is Shootout'] == 0].copy()
ols_elo  = sm.OLS(df_reg['Goal Diff'], sm.add_constant(df_reg[['Elo Diff']])).fit()
ols_fifa = sm.OLS(df_reg['Goal Diff'], sm.add_constant(df_reg[['Fifa Rank Diff']])).fit()

# ── Verification ───────────────────────────────────────────────────────────
print(f"\nDataset: {len(df)} matches | {df['Wc Year Int'].nunique()} tournaments (1994–2022)")
print(f"Shootouts: {df['Is Shootout'].sum()} | Extra time: {int(df['Extra Time'].sum())}")
print()
print("Quick-check vs known values:")
checks = [
    ("ELO accuracy",   elo_acc,                     0.739),
    ("FIFA accuracy",  fifa_acc,                     0.687),
    ("AUC ELO",        auc_elo,                      0.775),
    ("AUC FIFA",       auc_fifa,                     0.695),
    ("Brier ELO",      brier_elo,                    0.189),
    ("Pseudo-R² ELO",  pseudo_r2_elo,                0.188),
]
for name, got, expected in checks:
    ok = abs(got - expected) < 0.003
    print(f"  {'✅' if ok else '❌'} {name:<20} {got:.4f}  (expected ≈{expected})")
print()
print("All green? Run the chapters below in order.")

---
## Chapter 1 — The Data

**What we're looking at:** 115 World Cup knockout matches played across eight tournaments from USA 1994 to Qatar 2022. The 13 excluded matches lacked either an ELO or FIFA ranking value.

For each match we have:
- The **ELO rating** of both teams immediately before the match (from eloratings.net — validated against saved MHTML source pages)
- The **FIFA ranking** of both teams from the most recent snapshot before the tournament (from Jürisöö (2023) Kaggle dataset — real `rank_date` values, not proxy estimates)
- The **actual result** including whether the match went to extra time or penalties

The chart below shows three things: how spread out the ELO differences are, how compressed the FIFA rank differences are, and how fresh the ranking data was for each tournament.

In [ ]:
print("=== DATASET OVERVIEW ===")
print(f"Date range:    {df['Match Date'].min().date()} to {df['Match Date'].max().date()}")
print(f"Tournaments:   {df['Wc Year Int'].nunique()} World Cups")
print(f"Total matches: {len(df)}")
print(f"Shootouts:     {df['Is Shootout'].sum()} ({df['Is Shootout'].mean():.1%})")
print(f"Extra time:    {int(df['Extra Time'].sum())} matches")
print()

desc = df[['Elo Diff','Fifa Rank Diff','Goal Diff','Days Stale']].describe().round(2)
print("Descriptive statistics:")
print(desc.to_string())
print()
agree = (df['Elo Favourite'] == df['Fifa Favourite']).sum()
print(f"ELO and FIFA agree on the favourite: {agree}/{len(df)} matches ({agree/len(df):.1%})")
print(f"Discordant pairs (disagree):         {len(df)-agree} matches — these are what McNemar's test uses")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ELO differences
axes[0].hist(df['Elo Diff'], bins=25, color=ELO_COLOR, alpha=0.85, edgecolor='white')
axes[0].axvline(0, color='red', ls='--', lw=2, label='Even match')
axes[0].axvline(df['Elo Diff'].mean(), color=NAVY, lw=2, label=f"Mean={df['Elo Diff'].mean():.0f}")
axes[0].set_title('ELO Differences\n(Home − Away | positive = home stronger)', fontweight='bold')
axes[0].set_xlabel('ELO Difference'); axes[0].set_ylabel('Number of Matches')
axes[0].legend(fontsize=9)

# FIFA rank differences
axes[1].hist(df['Fifa Rank Diff'], bins=20, color=FIFA_COLOR, alpha=0.85, edgecolor='white')
axes[1].axvline(0, color='red', ls='--', lw=2, label='Even match')
axes[1].axvline(df['Fifa Rank Diff'].mean(), color=NAVY, lw=2, label=f"Mean={df['Fifa Rank Diff'].mean():.1f}")
axes[1].set_title('FIFA Rank Differences\n(Away − Home rank | positive = home better ranked)', fontweight='bold')
axes[1].set_xlabel('FIFA Rank Difference'); axes[1].legend(fontsize=9)

# Staleness
axes[2].hist(df['Days Stale'], bins=15, color=NAVY, alpha=0.85, edgecolor='white')
for yr, grp in df.groupby('Wc Year Int'):
    axes[2].axvline(grp['Days Stale'].mean(), color='red', alpha=0.4, lw=1.2)
axes[2].set_title('FIFA Snapshot Staleness\n(Real rank_date values — not proxies)', fontweight='bold')
axes[2].set_xlabel('Days between FIFA snapshot and match date')
axes[2].text(0.97, 0.95, f"Mean={df['Days Stale'].mean():.0f}d\nMax={df['Days Stale'].max():.0f}d",
             transform=axes[2].transAxes, ha='right', va='top', fontsize=9,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Distribution of Rating Differences | WC Knockout 1994–2022',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig1_distributions.png")

---
## Chapter 2 — The Core Finding

**The short version:** ELO outperforms FIFA on every metric.

But one metric is not enough. A single accuracy number can be a fluke. So we measure performance six different ways — accuracy, discrimination (AUC), calibration (Brier score), goal difference explained (OLS R²), out-of-sample cross-validation, and controlled regression. Each metric asks a slightly different question. If ELO wins on all six, that is convergent evidence that the result is real.

---
### 2.1 — Baseline Accuracy and Model Fit

In [ ]:
print("=== BASELINE LOGISTIC REGRESSION ===")
print("=" * 52)
print(f"{'Metric':<30} {'ELO':>10} {'FIFA':>10}")
print("-" * 52)
print(f"{'Accuracy':<30} {elo_acc:>10.1%} {fifa_acc:>10.1%}")
print(f"{'AUC-ROC':<30} {auc_elo:>10.4f} {auc_fifa:>10.4f}")
print(f"{'Coefficient':<30} {sm_elo.params[0]:>10.4f} {sm_fifa.params[0]:>10.4f}")
print(f"{'p-value (coefficient)':<30} {sm_elo.pvalues[0]:>10.4f} {sm_fifa.pvalues[0]:>10.4f}")
print(f"{'Pseudo-R² (50/50 null)':<30} {pseudo_r2_elo:>10.4f} {pseudo_r2_fifa:>10.4f}")
print()
print(f"ELO picks the right winner {elo_acc:.1%} of the time.")
print(f"FIFA picks the right winner {fifa_acc:.1%} of the time.")
print(f"That is a {elo_acc-fifa_acc:.1%} gap in favour of ELO.")
print()
print(f"ELO Pseudo-R² ({pseudo_r2_elo:.3f}) is {pseudo_r2_elo/pseudo_r2_fifa:.1f}x FIFA's ({pseudo_r2_fifa:.3f})")
print("against a 50/50 baseline — the correct null at neutral venues.")

# ROC curves
fpr_elo,  tpr_elo,  _ = roc_curve(y, y_prob_elo)
fpr_fifa, tpr_fifa, _ = roc_curve(y, y_prob_fifa)
fpr_pts,  tpr_pts,  _ = roc_curve(y, y_prob_pts)

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(fpr_elo,  tpr_elo,  color=ELO_COLOR,  lw=2.5, label=f'ELO (AUC = {auc_elo:.3f})')
ax.plot(fpr_fifa, tpr_fifa, color=FIFA_COLOR, lw=2.5, ls='--', label=f'FIFA Rank (AUC = {auc_fifa:.3f})')
ax.plot(fpr_pts,  tpr_pts,  color='#7C3AED',  lw=2.0, ls='-.', label=f'FIFA Points (AUC = {auc_pts:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1.2, label='Random (AUC = 0.500)')
ax.fill_between(fpr_elo, tpr_elo, alpha=0.08, color=ELO_COLOR)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves: ELO vs FIFA Rank vs FIFA Points\nWC Knockout 1994–2022',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('fig2_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig2_roc_curves.png")
print()
print("Note: FIFA Points (AUC=0.737) beats FIFA Rank (0.695) but still trails ELO (0.775).")
print("ELO beats FIFA even when FIFA is given its best possible predictor.")

---
### 2.2 — Calibration: Do Predicted Probabilities Match Reality?

Accuracy only tests whether the right team was picked — not *how confident* the model was. A system that says "60% chance of winning" for every match is useless even if it picks right most of the time.

The Brier score measures calibration: the lower, the better. A random coin-flip gives 0.250. Perfect prediction gives 0.000.

In [ ]:
print(f"Brier score — ELO:  {brier_elo:.4f}  ({(1-brier_elo/0.25):.1%} better than random)")
print(f"Brier score — FIFA: {brier_fifa:.4f}  ({(1-brier_fifa/0.25):.1%} better than random)")
print(f"Random baseline:    0.2500")
print()
print("ELO's probability estimates are better calibrated.")
print("FIFA shows a characteristic kink in the 0.50–0.70 range:")
print("it systematically underestimates the favourite's win probability")
print("in near-even contests — a consequence of its 600 divisor vs ELO's 400.")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, probs, label, color, brier in [
    (axes[0], y_prob_elo,  'ELO Model',  ELO_COLOR,  brier_elo),
    (axes[1], y_prob_fifa, 'FIFA Model', FIFA_COLOR, brier_fifa),
]:
    fop, mpv = calibration_curve(y, probs, n_bins=8, strategy='quantile')
    ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect calibration')
    ax.plot(mpv, fop, 'o-', color=color, lw=2.5, ms=8,
            label=f'{label} (Brier={brier:.4f})')
    ax.fill_between(mpv, mpv, fop, alpha=0.15, color=color, label='Calibration gap')
    ax.set_xlabel('Predicted Probability of Home Win', fontsize=10)
    ax.set_ylabel('Actual Win Rate in Probability Bin', fontsize=10)
    ax.set_title(f'{label} Calibration\nBrier Score: {brier:.4f}  (0=perfect | 0.25=random)',
                 fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Calibration: Do Predicted Probabilities Match Reality?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_calibration.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig8_calibration.png")

---
### 2.3 — Goal Difference: Does the Size of the Rating Gap Predict the Margin of Victory?

Binary win/loss is coarse. A team that wins 5-0 is not the same as a team that edges 1-0 in extra time. We test whether a larger rating gap predicts a larger winning margin — using only the 89 non-shootout matches (penalties add random variation that would corrupt the signal).

In [ ]:
print(f"Non-shootout matches: {len(df_reg)} (excluded {len(df)-len(df_reg)} shootout matches)")
print()
print(f"{'Metric':<30} {'ELO':>10} {'FIFA':>10}")
print("-" * 52)
print(f"{'R² (variance explained)':<30} {ols_elo.rsquared:>10.4f} {ols_fifa.rsquared:>10.4f}")
print(f"{'Coefficient (slope)':<30} {ols_elo.params['Elo Diff']:>10.4f} {ols_fifa.params['Fifa Rank Diff']:>10.4f}")
print(f"{'p-value':<30} {ols_elo.pvalues['Elo Diff']:>10.4f} {ols_fifa.pvalues['Fifa Rank Diff']:>10.4f}")
print(f"{'RMSE':<30} {np.sqrt(ols_elo.mse_resid):>10.4f} {np.sqrt(ols_fifa.mse_resid):>10.4f}")
print()
print(f"ELO explains {ols_elo.rsquared:.1%} of goal difference variance.")
print(f"FIFA explains {ols_fifa.rsquared:.1%} — ELO explains {ols_elo.rsquared/ols_fifa.rsquared:.0%} as much.")
print("This is expected: ELO updates ratings using goal margins,")
print("so it captures quality gaps more precisely than FIFA rank.")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, col, ols_m, color, label in [
    (axes[0], 'Elo Diff',       ols_elo,  ELO_COLOR, 'ELO'),
    (axes[1], 'Fifa Rank Diff', ols_fifa, FIFA_COLOR, 'FIFA Rank'),
]:
    ax.scatter(df_reg[col], df_reg['Goal Diff'], alpha=0.35, color=color, s=40)
    x_line = np.linspace(df_reg[col].min(), df_reg[col].max(), 100)
    ax.plot(x_line, ols_m.params.iloc[0] + ols_m.params.iloc[1]*x_line,
            '-', color='red', lw=2.5, label='Regression line')
    ax.axhline(0, color='gray', ls='--', lw=1)
    ax.axvline(0, color='gray', ls='--', lw=1)
    ax.set_xlabel(f'{label} Difference', fontsize=10)
    ax.set_ylabel('Goal Difference (Home−Away, FT/AET)', fontsize=10)
    ax.set_title(f'{label} vs Goal Difference\nR²={ols_m.rsquared:.4f} | p={ols_m.pvalues.iloc[1]:.4f}',
                 fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Rating Difference vs Goal Difference (Non-Shootout Matches)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig4_goal_diff_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig4_goal_diff_regression.png")

---
### 2.4 — Cross-Validation: Does the Advantage Hold Out-of-Sample?

We test each system on one held-out tournament at a time, trained on the other seven. This tests whether ELO's advantage is stable across different eras and tournament conditions — or just a product of the specific 115 matches in this dataset.

**Method note:** CV accuracy is computed directly as `Elo Correct.mean()` on the held-out tournament. No logistic model is needed for this — we are testing the *rating system*, not a trained classifier.

In [ ]:
tournaments = sorted(df['Wc Year Int'].unique())
elo_cv, fifa_cv, cv_years = [], [], []

print("Leave-One-Tournament-Out Cross-Validation")
print("=" * 52)
print(f"{'Year':<8} {'n_train':>8} {'n_test':>8} {'ELO':>10} {'FIFA':>10} {'Gap':>8}")
print("-" * 52)

for test_year in tournaments:
    train = df[df['Wc Year Int'] != test_year]
    test  = df[df['Wc Year Int'] == test_year]
    acc_elo_cv  = test['Elo Correct'].mean()
    acc_fifa_cv = test['Fifa Correct'].mean()
    elo_cv.append(acc_elo_cv)
    fifa_cv.append(acc_fifa_cv)
    cv_years.append(test_year)
    gap = acc_elo_cv - acc_fifa_cv
    flag = "← ELO wins" if gap > 0 else "← FIFA wins" if gap < 0 else "← tied"
    print(f"  {test_year:<6} {len(train):>8} {len(test):>8} {acc_elo_cv:>10.1%} {acc_fifa_cv:>10.1%} {gap:>+8.1%} {flag}")

print("-" * 52)
print(f"  {'Mean':<22}         {np.mean(elo_cv):>10.1%} {np.mean(fifa_cv):>10.1%} {np.mean(elo_cv)-np.mean(fifa_cv):>+8.1%}")
print(f"  {'SD':<22}         {np.std(elo_cv):>10.1%} {np.std(fifa_cv):>10.1%}")
print()
elo_wins_cv = sum(1 for e,f in zip(elo_cv,fifa_cv) if e>f)
print(f"ELO outperforms FIFA in {elo_wins_cv}/8 held-out tournaments.")
print(f"In-sample accuracy:   ELO {elo_acc:.1%} | FIFA {fifa_acc:.1%}")
print(f"Out-of-sample (CV):   ELO {np.mean(elo_cv):.1%} | FIFA {np.mean(fifa_cv):.1%}")
print(f"Stability gap:        ELO {elo_acc-np.mean(elo_cv):+.1%} | FIFA {fifa_acc-np.mean(fifa_cv):+.1%}")
print()
print("2002 note: South Korea's semi-final run was unforeseeable from historical data.")
print("FIFA actually beats ELO in that fold — an honest result, not a bug.")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
x = np.arange(len(cv_years)); w = 0.35
axes[0].bar(x-w/2, elo_cv,  w, color=ELO_COLOR,  alpha=0.85, label=f'ELO mean={np.mean(elo_cv):.1%}')
axes[0].bar(x+w/2, fifa_cv, w, color=FIFA_COLOR, alpha=0.85, label=f'FIFA mean={np.mean(fifa_cv):.1%}')
axes[0].axhline(np.mean(elo_cv),  color=ELO_COLOR,  ls='--', lw=2)
axes[0].axhline(np.mean(fifa_cv), color=FIFA_COLOR, ls='--', lw=2)
axes[0].axhline(0.5, color='red', ls=':', lw=1.5, label='Chance 50%')
axes[0].set_xticks(x); axes[0].set_xticklabels(cv_years)
axes[0].set_ylabel('CV Accuracy'); axes[0].legend(fontsize=9)
axes[0].set_title('CV Accuracy Per Held-Out Tournament', fontweight='bold')
axes[0].set_ylim(0, 1.0)

cats = ['ELO\nIn-Sample','ELO\nOut-of-Sample','FIFA\nIn-Sample','FIFA\nOut-of-Sample']
vals = [elo_acc, np.mean(elo_cv), fifa_acc, np.mean(fifa_cv)]
cols = [ELO_COLOR,'#93C5FD',FIFA_COLOR,'#FCD34D']
bars = axes[1].bar(cats, vals, color=cols, edgecolor='white', width=0.5)
for bar, val in zip(bars, vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.01,
                 f'{val:.1%}', ha='center', fontweight='bold', fontsize=12)
axes[1].axhline(0.5, color='red', ls='--', lw=1.5)
axes[1].set_ylim(0.3, 0.9)
axes[1].set_title('In-Sample vs Out-of-Sample', fontweight='bold')

plt.suptitle('Leave-One-Tournament-Out Cross-Validation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig9_cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig9_cross_validation.png")

---
### 2.5 — Controlled Regression: Is the Gap Real Once We Account for Round, Year, and Era?

ELO might appear better simply because quarter-finals are less predictable than semi-finals, or because recent tournaments are different from older ones. This model adds those controls and asks whether ELO's advantage survives.

In [ ]:
df_model = df.dropna(subset=['round_num']).copy()
y_ctrl   = df_model['home_won'].values

X_elo_ctrl  = sm.add_constant(df_model[['Elo Diff','round_num','year_centered','post_2018']])
X_fifa_ctrl = sm.add_constant(df_model[['Fifa Rank Diff','round_num','year_centered','post_2018']])

m_elo_ctrl  = sm.Logit(y_ctrl, X_elo_ctrl).fit(disp=0)
m_fifa_ctrl = sm.Logit(y_ctrl, X_fifa_ctrl).fit(disp=0)

print("CONTROLLED LOGISTIC REGRESSION")
print("=" * 68)
print(f"{'Variable':<25} {'ELO Coef':>10} {'p':>8}    {'FIFA Coef':>10} {'p':>8}")
print("-" * 68)
labels    = ['Rating difference','Round (ordinal)','Year (centred)','Post-2018 dummy']
vars_elo  = ['Elo Diff','round_num','year_centered','post_2018']
vars_fifa = ['Fifa Rank Diff','round_num','year_centered','post_2018']
for lbl, ve, vf in zip(labels, vars_elo, vars_fifa):
    ce = m_elo_ctrl.params.get(ve, np.nan)
    pe = m_elo_ctrl.pvalues.get(ve, np.nan)
    cf = m_fifa_ctrl.params.get(vf, np.nan)
    pf = m_fifa_ctrl.pvalues.get(vf, np.nan)
    sig_e = "***" if pe < 0.001 else "**" if pe < 0.01 else "*" if pe < 0.05 else ""
    print(f"  {lbl:<23} {ce:>10.4f} {pe:>8.4f}{sig_e:<3}  {cf:>10.4f} {pf:>8.4f}")
print()
print(f"  {'Pseudo-R²':<23} {m_elo_ctrl.prsquared:>10.4f}           {m_fifa_ctrl.prsquared:>10.4f}")
print()
print("Key finding: ELO's coefficient stays significant (p<0.001) after all controls.")
print("The post-2018 dummy is non-significant in both models (p>0.70).")
print("This means the 2018 formula change worked through rating differences,")
print("not through some independent structural shift in predictability.")
print()
print("--- 95% Confidence Intervals (verify against paper Tables 7 & 12) ---")
print("ELO controlled model 95% CIs:")
print(m_elo_ctrl.conf_int().round(4).to_string())
print()
print("FIFA controlled model 95% CIs:")
print(m_fifa_ctrl.conf_int().round(4).to_string())
print()
print("Paper claims: ELO rating diff CI = [0.0037, 0.0116] | FIFA rating diff CI = [0.0134, 0.0702]")

---
## Chapter 3 — Why McNemar's Test Didn't Reach Significance

We have six metrics all pointing the same direction. But the formal test — McNemar's — gave p = 0.180. Does this mean the gap might just be noise?

No. It means the test doesn't have enough data to detect the gap, even if the gap is real. This chapter shows why.

---
### 3.1 — The McNemar Contingency Table

In [ ]:
b = int(((df['Elo Correct']==1) & (df['Fifa Correct']==0)).sum())
c = int(((df['Elo Correct']==0) & (df['Fifa Correct']==1)).sum())
aa = int((df['Elo Correct'] & df['Fifa Correct']).sum())
dd = int(((df['Elo Correct']==0) & (df['Fifa Correct']==0)).sum())
chi2_val  = (b-c)**2 / (b+c)
p_mcnemar = float(scipy_stats.chi2.sf(chi2_val, 1))

print("McNemar's test looks only at the matches where the two systems disagreed.")
print("The 95 matches where both systems agreed carry ZERO information here.")
print()
print("Contingency table:")
print(f"                       FIFA ✓   FIFA ✗")
print(f"  ELO ✓                  {aa:>3}      {b:>3}   ← ELO right, FIFA wrong  (b={b})")
print(f"  ELO ✗                  {c:>3}      {dd:>3}   ← FIFA right, ELO wrong  (c={c})")
print()
print(f"  Discordant pairs: b={b}, c={c}  (ratio b/c = {b/c:.2f} — >1 favours ELO)")
print(f"  Chi² = ({b}−{c})²/({b}+{c}) = {chi2_val:.4f}")
print(f"  p-value = {p_mcnemar:.4f}")
print()
print(f"  Result: NOT significant at α=0.05")
print()
print("  This does NOT mean the systems are equivalent.")
print("  It means the test cannot distinguish them with only 20 discordant pairs.")
print("  See the power analysis below for why this is a data constraint, not a null result.")

---
### 3.2 — Statistical Power: How Much Data Would We Need?

McNemar's test works on discordant pairs only (b+c=20). The correct power formula uses those 20 pairs, not the total 115 matches.

In [ ]:
p_elo_v  = df['Elo Correct'].mean()
p_fifa_v = df['Fifa Correct'].mean()
cohens_h = abs(2*np.arcsin(np.sqrt(p_elo_v)) - 2*np.arcsin(np.sqrt(p_fifa_v)))

b_pow = b; c_pow = c
n_d   = b_pow + c_pow
p1    = b_pow / n_d
z_alpha  = norm_dist.ppf(0.975)
z_effect = (p1 - 0.5) / np.sqrt(0.25 / n_d)
power    = norm_dist.cdf(z_effect - z_alpha)

disc_rate = n_d / len(df)
for n_d_test in range(5, 5000):
    z_e = (p1 - 0.5) / np.sqrt(0.25 / n_d_test)
    if norm_dist.cdf(z_e - z_alpha) >= 0.80:
        n_d_needed = n_d_test
        break
n_needed = int(np.ceil(n_d_needed / disc_rate))

print(f"Effect size (Cohen's h): {cohens_h:.4f} — Small effect")
print()
print(f"McNemar power analysis (correct formula — discordant pairs only):")
print(f"  ELO wins {p1:.0%} of the {n_d} discordant pairs (H0 would be 50%)")
print(f"  Statistical power at n_d={n_d}: {power:.1%}")
print(f"  → This test would MISS the effect {1-power:.1%} of the time even if ELO is genuinely better")
print()
print(f"  To reach 80% power:")
print(f"  Need {n_d_needed} discordant pairs")
print(f"  At the current discordant rate ({disc_rate:.1%}), that implies ~{n_needed} total matches")
print(f"  Equivalent to ~{n_needed//16} more World Cup tournaments")
print()
print("The study is structurally underpowered — not because of methodology,")
print("but because World Cup knockout stages only produce ~15 matches every 4 years.")

ns = np.arange(20, 750, 5)
powers = []
for n_total in ns:
    n_d_i = n_total * disc_rate
    z_e_i = (p1 - 0.5) / np.sqrt(0.25 / max(n_d_i, 1))
    powers.append(norm_dist.cdf(z_e_i - z_alpha))

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(ns, powers, lw=2.5, color=NAVY)
ax.fill_between(ns, powers, alpha=0.1, color=NAVY)
ax.axhline(0.8, color='green', ls='--', lw=2, label='80% power threshold')
ax.axhline(0.05, color='red', ls='--', lw=1.5, label='Type I error (α=0.05)')
ax.axvline(len(df), color='orange', ls=':', lw=2,
           label=f'Current study (n={len(df)}, power={power:.0%})')
ax.axvline(n_needed, color='green', ls=':', lw=2,
           label=f'Required sample (n={n_needed}, power=80%)')
ax.set_xlabel('Total Knockout Matches', fontsize=11)
ax.set_ylabel('Statistical Power', fontsize=11)
ax.set_title(f'Statistical Power vs Sample Size (Effect size h={cohens_h:.3f})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('fig7_power_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig7_power_curve.png")

---
### 3.3 — Bootstrap: The Most Informative Single Statistic

Because McNemar's is underpowered, the bootstrap provides more useful information. We resample 10,000 times and ask: what fraction of resamples show ELO ahead?

In [ ]:
np.random.seed(42)
elo_boot, fifa_boot, gap_boot = [], [], []
for _ in range(10000):
    sample = df.sample(n=len(df), replace=True)
    e = sample['Elo Correct'].mean()
    f = sample['Fifa Correct'].mean()
    elo_boot.append(e); fifa_boot.append(f); gap_boot.append(e-f)

elo_boot  = np.array(elo_boot)
fifa_boot = np.array(fifa_boot)
gap_boot  = np.array(gap_boot)
pct_elo_wins = (gap_boot > 0).mean()
gap_ci95 = (np.percentile(gap_boot, 2.5), np.percentile(gap_boot, 97.5))
elo_ci95  = (np.percentile(elo_boot, 2.5), np.percentile(elo_boot, 97.5))
fifa_ci95 = (np.percentile(fifa_boot, 2.5), np.percentile(fifa_boot, 97.5))

print("Bootstrap (10,000 resamples, paired — same sample for both systems)")
print("=" * 60)
print(f"ELO  accuracy: {elo_acc:.1%}   95% CI: [{elo_ci95[0]:.1%}, {elo_ci95[1]:.1%}]")
print(f"FIFA accuracy: {fifa_acc:.1%}   95% CI: [{fifa_ci95[0]:.1%}, {fifa_ci95[1]:.1%}]")
print(f"Gap (ELO−FIFA): {elo_acc-fifa_acc:+.1%}  95% CI: [{gap_ci95[0]:+.1%}, {gap_ci95[1]:+.1%}]")
print()
print(f"ELO exceeded FIFA in {pct_elo_wins:.1%} of all 10,000 bootstrap resamples.")
print()
print("The 95% CI includes zero — consistent with McNemar non-significance.")
print(f"But {pct_elo_wins:.1%} directional signal across resamples is the key finding.")
print()
ci_results = {}
for level, tail in [(90, 5.0), (95, 2.5), (99, 0.5)]:
    lo = np.percentile(gap_boot, tail)
    hi = np.percentile(gap_boot, 100-tail)
    ci_results[level] = (lo, hi)
    sig = "NOT significant" if lo < 0 else "Significant"
    print(f"  {level}% CI: [{lo:+.1%}, {hi:+.1%}]  {sig} — CI {'excludes' if lo>0 else 'includes'} zero")

fig = plt.figure(figsize=(18, 9))
ax1 = fig.add_subplot(2,3,1); ax2 = fig.add_subplot(2,3,2)
ax3 = fig.add_subplot(2,3,3); ax4 = fig.add_subplot(2,1,2)
obs_gap = elo_acc - fifa_acc
for ax, data, obs, color, title in [
    (ax1, elo_boot,  elo_acc,  ELO_COLOR, f'ELO Accuracy\n{elo_acc:.1%}'),
    (ax2, fifa_boot, fifa_acc, FIFA_COLOR, f'FIFA Accuracy\n{fifa_acc:.1%}'),
    (ax3, gap_boot,  obs_gap,  '#16A34A',  f'Gap (ELO−FIFA)\n{obs_gap:+.1%}'),
]:
    ax.hist(data, bins=60, color=color, alpha=0.80, edgecolor='white', lw=0.3)
    ax.axvline(obs, color='black', lw=2.2)
    lo95, hi95 = np.percentile(data, [2.5, 97.5])
    ax.axvline(lo95, color='red', ls='--', lw=1.8, label='95% CI')
    ax.axvline(hi95, color='red', ls='--', lw=1.8)
    if ax is ax3: ax.axvline(0, color='purple', ls=':', lw=2, label='Zero')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8)

for level, (lo, hi) in {90:ci_results[90],95:ci_results[95],99:ci_results[99]}.items():
    yp = {90:2,95:1,99:0}[level]
    c  = {90:'#16A34A',95:'#2563EB',99:'#DC2626'}[level]
    ax4.barh(yp, hi-lo, left=lo, height=0.4, color=c, alpha=0.75)
    ax4.text(hi+0.003, yp, f'[{lo:+.1%}, {hi:+.1%}]  Includes zero', va='center', fontsize=10)
ax4.axvline(0,       color='black', lw=2.0, ls='--', label='Zero (no difference)')
ax4.axvline(obs_gap, color='black', lw=2.5, label=f'Observed gap ({obs_gap:+.1%})')
ax4.set_yticks([0,1,2])
ax4.set_yticklabels(['99% CI (strict)','95% CI (standard)','90% CI (relaxed)'], fontsize=11)
ax4.set_title('Sensitivity Analysis: ELO−FIFA Gap CI at 90%, 95%, 99%', fontweight='bold')
ax4.legend(fontsize=10)
plt.suptitle('Bootstrap Distributions & Multi-Level CI Sensitivity Analysis',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig6_bootstrap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig6_bootstrap.png")

---
## Chapter 4 — Does the Gap Hold Over Time?

FIFA explicitly reformed its ranking formula in 2018, adopting ELO-based logic. Did this close the gap?

The short answer: it narrowed the gap from +6.0pp to +3.2pp — but did not close it. FIFA's 2006 collapse to 37.5% accuracy (below chance) was the most dramatic evidence of its pre-2018 formula's flaw. ELO held at 62.5% that year.

In [ ]:
acc_by_year = df.groupby('Wc Year Int').agg(
    elo_acc=('Elo Correct','mean'),
    fifa_acc=('Fifa Correct','mean'),
    elo_upset=('elo_upset','mean'),
    fifa_upset=('fifa_upset','mean'),
    n=('Elo Correct','count'),
    era=('post_2018','first')
).reset_index()
acc_by_year['gap'] = acc_by_year['elo_acc'] - acc_by_year['fifa_acc']

print("Per-tournament accuracy:")
print(f"{'Year':<8} {'ELO':>8} {'FIFA':>8} {'Gap':>8} {'n':>4}  Era")
print("-" * 55)
for _, row in acc_by_year.iterrows():
    era = "Post-2018" if row['era'] else "Pre-2018 "
    flag = "⚠️  FIFA below chance!" if row['fifa_acc'] < 0.50 else ""
    print(f"  {int(row['Wc Year Int']):<6} {row['elo_acc']:>8.1%} {row['fifa_acc']:>8.1%} "
          f"{row['gap']:>+8.1%} {int(row['n']):>4}  {era} {flag}")

pre  = df[df['post_2018']==0]
post = df[df['post_2018']==1]
print()
print("Era comparison:")
print(f"  Pre-2018  (n={len(pre)}): ELO {pre['Elo Correct'].mean():.1%}  FIFA {pre['Fifa Correct'].mean():.1%}  Gap {pre['Elo Correct'].mean()-pre['Fifa Correct'].mean():+.1%}")
print(f"  Post-2018 (n={len(post)}): ELO {post['Elo Correct'].mean():.1%}  FIFA {post['Fifa Correct'].mean():.1%}  Gap {post['Elo Correct'].mean()-post['Fifa Correct'].mean():+.1%}")
print()
print(f"Gap narrowed from {pre['Elo Correct'].mean()-pre['Fifa Correct'].mean():+.1%} to {post['Elo Correct'].mean()-post['Fifa Correct'].mean():+.1%} after 2018 reform.")
print("Post-2018 sample is only 31 matches across 2 tournaments.")
print("The narrowing is real but cannot be confirmed statistically at this sample size.")

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
axes[0].plot(acc_by_year['Wc Year Int'], acc_by_year['elo_acc'], 'o-',
             color=ELO_COLOR, lw=2.5, ms=8, label='ELO')
axes[0].plot(acc_by_year['Wc Year Int'], acc_by_year['fifa_acc'], 's-',
             color=FIFA_COLOR, lw=2.5, ms=8, label='FIFA Rankings')
for _, row in acc_by_year.iterrows():
    axes[0].annotate(f"n={int(row['n'])}",
                     (row['Wc Year Int'], max(row['elo_acc'],row['fifa_acc'])+0.018),
                     fontsize=8, ha='center')
axes[0].axhline(0.5, color='red', ls='--', lw=1.5, label='Chance (50%)')
axes[0].axvline(2018, color='green', ls=':', lw=2, alpha=0.8)
axes[0].text(2018.3, 0.36, 'FIFA adopts\nELO formula\n(2018)', fontsize=8, color='green')
axes[0].set_ylim(0.3, 1.0); axes[0].set_ylabel('Prediction Accuracy')
axes[0].set_title('Prediction Accuracy per Tournament', fontweight='bold')
axes[0].legend(fontsize=9)

x = np.arange(len(acc_by_year)); w = 0.35
axes[1].bar(x-w/2, acc_by_year['elo_upset'],  w, color=ELO_COLOR,  alpha=0.8, label='ELO Upset Rate')
axes[1].bar(x+w/2, acc_by_year['fifa_upset'], w, color=FIFA_COLOR, alpha=0.8, label='FIFA Upset Rate')
axes[1].axvline(6.5, color='green', ls=':', lw=2, alpha=0.8, label='FIFA formula change')
axes[1].text(6.6, 0.65, 'FIFA formula\nchange', fontsize=8, color='green')
axes[1].set_xticks(x); axes[1].set_xticklabels(acc_by_year['Wc Year Int'].astype(int))
axes[1].set_title('Upset Frequency (Lower = more predictive)', fontweight='bold')
axes[1].set_ylabel('Upset Rate'); axes[1].legend(fontsize=9); axes[1].set_ylim(0, 0.8)

plt.suptitle('ELO vs FIFA Prediction Accuracy Over Time | WC Knockout 1994–2022',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_accuracy_over_time.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig3_accuracy_over_time.png")

---
## Chapter 5 — Three Robustness Checks

Before concluding, we stress-test the core finding three ways:

1. **Staleness** — Were FIFA's rankings just stale at some tournaments? Does freshness explain the gap?
2. **Exclude 1994** — The 1994 tournament had the freshest rankings (24 days stale). Does removing it change anything?
3. **FIFA Points vs FIFA Rank** — Rank is ordinal and compressed. FIFA points carry more continuous information. Does ELO still win?

In [ ]:
# ── Robustness 1: Staleness ───────────────────────────────────────────────
print("ROBUSTNESS CHECK 1 — STALENESS")
print("=" * 50)
stale = df.groupby('Wc Year Int').agg(
    snapshot=('Fifa Snapshot Date','first'),
    min_stale=('Days Stale','min'),
    max_stale=('Days Stale','max'),
    mean_stale=('Days Stale','mean')
).reset_index()

print(f"{'Year':<8} {'Snapshot':<14} {'Min':>6} {'Max':>6} {'Mean':>8}")
print("-" * 48)
for _, row in stale.iterrows():
    print(f"  {int(row['Wc Year Int']):<6} {str(row['snapshot'])[:10]:<14} "
          f"{row['min_stale']:>6.0f} {row['max_stale']:>6.0f} {row['mean_stale']:>8.1f}d")

df_stale = df.dropna(subset=['round_num','Days Stale']).copy()
y_stale  = df_stale['home_won'].values
X_stale  = sm.add_constant(df_stale[['Fifa Rank Diff','round_num',
                                      'year_centered','post_2018','Days Stale']])
m_stale  = sm.Logit(y_stale, X_stale).fit(disp=0)

print()
print(f"Staleness regression coefficient: {m_stale.params['Days Stale']:.5f}")
print(f"Staleness p-value:                {m_stale.pvalues['Days Stale']:.4f}")
print()
print(f"Result: p={m_stale.pvalues['Days Stale']:.4f} — NOT significant.")
print("FIFA's inferiority is structural — not caused by stale ranking data.")
print("This holds across all 8 tournaments using real rank_date values.")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = ['#DC2626' if y==1994 else '#D97706' if y==2022 else ELO_COLOR
          for y in stale['Wc Year Int']]
bars = axes[0].bar(stale['Wc Year Int'].astype(str), stale['mean_stale'],
                   color=colors, alpha=0.85, edgecolor='white')
for bar, (_, row) in zip(bars, stale.iterrows()):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f"{row['mean_stale']:.0f}d", ha='center', fontsize=9, fontweight='bold')
axes[0].set_title('FIFA Ranking Staleness by Tournament\n(Real rank_date — not proxy values)',
                  fontweight='bold')
axes[0].set_ylabel('Mean Days Stale')

bins = pd.cut(df_stale['Days Stale'], bins=8)
stale_acc = df_stale.groupby(bins, observed=True)['Fifa Correct'].mean()
mid_pts = [i.mid for i in stale_acc.index]
axes[1].scatter(df_stale['Days Stale'], df_stale['Fifa Correct'], alpha=0.25, color=FIFA_COLOR, s=35)
axes[1].plot(mid_pts, stale_acc.values, 'o-', color='red', lw=2.5, ms=8, label='Binned mean')
axes[1].axhline(df_stale['Fifa Correct'].mean(), color='gray', ls='--',
                label=f'Overall mean ({df_stale["Fifa Correct"].mean():.1%})')
axes[1].set_xlabel('Days Stale'); axes[1].set_ylabel('FIFA Correct')
axes[1].set_title(f'FIFA Accuracy vs Staleness\n(p={m_stale.pvalues["Days Stale"]:.3f} — flat, non-significant)',
                  fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('FIFA Ranking Staleness Analysis (Real rank_date Data)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_s3_staleness_real.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_s3_staleness_real.png")

In [ ]:
# ── Robustness 2: Exclude 1994 ────────────────────────────────────────────
print("ROBUSTNESS CHECK 2 — EXCLUDE 1994")
print("=" * 50)
df_no94 = df[df['Wc Year Int'] != 1994].copy()
print(f"Full sample: {len(df)} | Excluding 1994: {len(df_no94)}")
print()

results_94 = []
for label, subset in [('Full (n=115)', df), ('Excl. 1994 (n=101)', df_no94)]:
    y_s = subset['home_won'].values
    m_e = LogisticRegression(fit_intercept=False, random_state=42).fit(subset[['Elo Diff']], y_s)
    m_f = LogisticRegression(fit_intercept=False, random_state=42).fit(subset[['Fifa Rank Diff']], y_s)
    sm_e = sm.Logit(y_s, subset[['Elo Diff']].values).fit(disp=0)
    sm_f = sm.Logit(y_s, subset[['Fifa Rank Diff']].values).fit(disp=0)
    results_94.append({
        'label':   label,
        'elo_acc': subset['Elo Correct'].mean(),
        'fifa_acc':subset['Fifa Correct'].mean(),
        'gap':     subset['Elo Correct'].mean() - subset['Fifa Correct'].mean(),
        'auc_elo': roc_auc_score(y_s, m_e.predict_proba(subset[['Elo Diff']])[:,1]),
        'auc_fifa':roc_auc_score(y_s, m_f.predict_proba(subset[['Fifa Rank Diff']])[:,1]),
    })

print(f"{'Metric':<25} {'Full':>14} {'Excl. 1994':>14} {'Change':>8}")
print("-" * 65)
for metric, key in [('ELO accuracy','elo_acc'),('FIFA accuracy','fifa_acc'),
                    ('Gap','gap'),('AUC ELO','auc_elo'),('AUC FIFA','auc_fifa')]:
    fv = results_94[0][key]; nv = results_94[1][key]
    d  = '↑' if nv > fv else '↓' if nv < fv else '='
    print(f"  {metric:<23} {fv:>14.4f} {nv:>14.4f} {d:>8}")
print()
print("Verdict: ELO leads FIFA on all metrics with or without 1994.")
print(f"The accuracy gap barely moves: {results_94[0]['gap']:+.1%} → {results_94[1]['gap']:+.1%}.")
print("1994 is not driving the result.")

In [ ]:
# ── Robustness 3: FIFA Points vs FIFA Rank ────────────────────────────────
print("ROBUSTNESS CHECK 3 — FIFA POINTS vs FIFA RANK")
print("=" * 55)

# Verify Fifa Pts Diff has no missing values
pts_nan = df['Fifa Pts Diff'].isna().sum()
print(f"NaN check — Fifa Pts Diff missing values: {pts_nan}")
if pts_nan > 0:
    print(f"WARNING: {pts_nan} rows have missing Fifa Pts Diff — those rows will be dropped in logistic fit.")
print()

results_pts = {}
for name, col, acc_val in [
    ('ELO',         'Elo Diff',       elo_acc),
    ('FIFA Rank',   'Fifa Rank Diff', fifa_acc),
    ('FIFA Points', 'Fifa Pts Diff',  None),
]:
    m_tmp  = LogisticRegression(fit_intercept=False, random_state=42).fit(df[[col]], y)
    probs  = m_tmp.predict_proba(df[[col]])[:,1]
    sm_tmp = sm.Logit(y, df[[col]].values).fit(disp=0)
    pr2    = 1 - sm_tmp.llf / ll_null_50
    df_r   = df[df['Is Shootout']==0]
    ols_m  = sm.OLS(df_r['Goal Diff'], sm.add_constant(df_r[[col]])).fit()
    if acc_val is None:
        acc_val = (df['Elo Correct'].mean() if col=='Elo Diff'
                   else df['Fifa Correct'].mean() if col=='Fifa Rank Diff'
                   else np.mean((df[[col]].values.flatten() > 0).astype(int) == df['home_won'].values))
    results_pts[name] = {
        'accuracy': acc_val,
        'auc':      roc_auc_score(y, probs),
        'brier':    np.mean((probs-y)**2),
        'goal_r2':  ols_m.rsquared,
        'pseudo_r2':pr2,
    }

# ── CV accuracy for all three systems ────────────────────────────────────
elo_cv_pts_list  = []
fifa_cv_pts_list = []
fifa_pts_cv_list = []

for test_year in sorted(df['Wc Year Int'].unique()):
    test = df[df['Wc Year Int'] == test_year]
    # ELO CV: already uses Elo Correct flag (same logic)
    elo_cv_pts_list.append(test['Elo Correct'].mean())
    # FIFA Rank CV: uses Fifa Correct flag
    fifa_cv_pts_list.append(test['Fifa Correct'].mean())
    # FIFA Points CV: predict home wins if Fifa Pts Diff > 0
    pts_pred_correct = ((test['Fifa Pts Diff'] > 0) == test['home_won'].astype(bool)).astype(int)
    fifa_pts_cv_list.append(pts_pred_correct.mean())

results_pts['ELO']['cv_accuracy']         = np.mean(elo_cv_pts_list)
results_pts['FIFA Rank']['cv_accuracy']   = np.mean(fifa_cv_pts_list)
results_pts['FIFA Points']['cv_accuracy'] = np.mean(fifa_pts_cv_list)

print(f"{'Metric':<25} {'ELO':>10} {'FIFA Rank':>12} {'FIFA Points':>13}")
print("-" * 63)
for metric in ['accuracy','auc','brier','goal_r2','pseudo_r2','cv_accuracy']:
    vals = [results_pts[s][metric] for s in ['ELO','FIFA Rank','FIFA Points']]
    better = '↓ better' if metric=='brier' else ''
    print(f"  {metric:<23} {vals[0]:>10.4f} {vals[1]:>12.4f} {vals[2]:>13.4f} {better}")
print()
pts_beats = sum(1 for m in ['accuracy','auc','goal_r2','pseudo_r2','cv_accuracy']
                if results_pts['FIFA Points'][m] > results_pts['FIFA Rank'][m])
pts_beats += (1 if results_pts['FIFA Points']['brier'] < results_pts['FIFA Rank']['brier'] else 0)
elo_leads = sum(1 for m in ['accuracy','auc','goal_r2','pseudo_r2','cv_accuracy']
                if results_pts['ELO'][m] > results_pts['FIFA Points'][m])
elo_leads += (1 if results_pts['ELO']['brier'] < results_pts['FIFA Points']['brier'] else 0)
print(f"FIFA Points beats FIFA Rank on {pts_beats}/6 metrics.")
print(f"ELO leads on {elo_leads}/6 metrics against FIFA Points.")
print()
print("Why use rank as the primary FIFA predictor?")
print("  (a) Rank is what FIFA actually uses for World Cup seeding draws")
print("  (b) Points are not comparable across eras: pre-2018 mean ≈626, post-2018 mean ≈1821")
print("  (c) Maintains comparability with prior literature")
print()
print("But even giving FIFA its best possible predictor (points), ELO still wins.")
print("This makes ELO's advantage more convincing, not less.")

---
## Chapter 6 — Complete Results Summary

All six metrics. All verified. Everything in one place.

In [ ]:
print("=" * 70)
print("COMPLETE RESULTS — FIFA vs ELO | WC Knockout 1994–2022")
print("=" * 70)
print()
print(f"Dataset: {len(df)} matches | 8 tournaments | 1994–2022")
print(f"         {df['Is Shootout'].sum()} decided by penalty shootout")
print()
print(f"{'Metric':<35} {'ELO':>10} {'FIFA':>10} {'ELO Wins':>10}")
print("-" * 67)
metrics_final = [
    ('Accuracy',              elo_acc,              fifa_acc,              True),
    ('AUC-ROC',               auc_elo,              auc_fifa,              True),
    ('Brier score (↓ better)',brier_elo,             brier_fifa,            False),
    ('Goal diff R²',          ols_elo.rsquared,     ols_fifa.rsquared,     True),
    ('CV accuracy',           np.mean(elo_cv),      np.mean(fifa_cv),      True),
    ('Pseudo-R² (controlled)',m_elo_ctrl.prsquared, m_fifa_ctrl.prsquared, True),
]
for name, ev, fv, higher_is_better in metrics_final:
    elo_wins = (ev < fv) if not higher_is_better else (ev > fv)
    flag = "✅ ELO" if elo_wins else "❌ FIFA"
    print(f"  {name:<33} {ev:>10.4f} {fv:>10.4f} {flag:>10}")

print()
print(f"McNemar:   b={b}, c={c}, p={p_mcnemar:.4f} | Power={power:.1%} | Need ~{n_needed} total matches")
print(f"Bootstrap: ELO>FIFA in {pct_elo_wins:.1%} resamples | 95% CI: [{gap_ci95[0]:+.1%}, {gap_ci95[1]:+.1%}]")
print()
print(f"ERA SPLIT:")
print(f"  Pre-2018  (n={len(pre)}, 6 tournaments): ELO {pre['Elo Correct'].mean():.1%} | FIFA {pre['Fifa Correct'].mean():.1%} | Gap {pre['Elo Correct'].mean()-pre['Fifa Correct'].mean():+.1%}")
print(f"  Post-2018 (n={len(post)}, 2 tournaments): ELO {post['Elo Correct'].mean():.1%} | FIFA {post['Fifa Correct'].mean():.1%} | Gap {post['Elo Correct'].mean()-post['Fifa Correct'].mean():+.1%}")
print()
print(f"STALENESS: p={m_stale.pvalues['Days Stale']:.4f} — not significant. Gap is structural, not artefactual.")
print()
print("=" * 70)
print("ELO outperforms FIFA on all six metrics.")
print("The gap survived three robustness checks.")
print("McNemar's non-significance reflects a power constraint, not system equivalence.")
print("=" * 70)

# Figure inventory
print()
print("Figures saved this session:")
import os
for fn in ['fig1_distributions.png','fig2_roc_curves.png','fig3_accuracy_over_time.png',
           'fig4_goal_diff_regression.png','fig6_bootstrap.png','fig7_power_curve.png',
           'fig8_calibration.png','fig9_cross_validation.png','fig_s3_staleness_real.png']:
    status = '✅' if os.path.exists(fn) else '⬜ not yet (run that chapter first)'
    print(f"  {status}  {fn}")